# Notebook 04 — Hybrid Safety & Governance Pipeline Demo

**Author:** Arbaz Khalid — Safety & Governance Module  
**Status:** End-to-end demo: report → Safety gate → Governance extraction → incident node

## Design
- Structured NYC 311 reports: Safety by category/descriptor rules; coordinates come from the report.
- Free text (social posts): Safety by the tweet-trained TF-IDF + logistic regression model (threshold 0.35, recall-first); Governance extracts location by regex.

## Known limitations
- The protocol lookup is a **placeholder dictionary**, not real RAG over NYC documents.
- Text-only reports have no coordinates yet (geocoding is a later step).
- Hazard type for text is assigned by keywords, which is rough.
- Severity is a simple heuristic.
- The text model is a baseline and misses some natural sentences.

In [1]:
import re
import json
import hashlib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

sf = pd.read_csv("data/processed/humaid_safety.csv")
HOLDOUT = ["maryland_floods_2018", "srilanka_floods_2017"]
train = sf[(~sf["event"].isin(HOLDOUT)) & (sf["split"] == "train")]

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=True)
clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(vec.fit_transform(train["text"]), train["safety_label"])
HZ = list(clf.classes_).index("hazard")
THRESHOLD = 0.35
print("Text model ready, threshold", THRESHOLD)

Text model ready, threshold 0.35


In [2]:
RULES_311 = [
    ("Sewer", "Sewer Backup", "flood"),
    ("Sewer", "Street Flooding", "flood"),
    ("Sewer", "Catch Basin", "flood"),
    ("Sewer", "Manhole Overflow", "flood"),
    ("Water System", "Excessive Water In Basement", "flood"),
    ("Damaged Tree", "Entire Tree Has Fallen", "road_blockage"),
    ("Damaged Tree", "Branch or Limb Has Fallen", "road_blockage"),
    ("Damaged Tree", "Tree Leaning/Uprooted", "road_blockage"),
    ("Street Condition", "Cave-in", "road_blockage"),
    ("Sewer", "Manhole Cover Missing", "road_blockage"),
]

def safety_311(ctype, descriptor):
    for rule_type, keyword, label in RULES_311:
        if ctype == rule_type and keyword.lower() in str(descriptor).lower():
            return label
    return "non_hazard"

def safety_text(text):
    return float(clf.predict_proba(vec.transform([text]))[0][HZ])

In [3]:
STREET_WORDS = r"(?:street|st|avenue|ave|boulevard|blvd|road|rd|drive|dr|parkway|pkwy|place|pl)"
ADDRESS_RE = re.compile(rf"\b\d{{1,5}}(?:-\d{{1,5}})?\s+(?:[A-Za-z0-9]+\s+){{0,3}}{STREET_WORDS}\b", re.I)
STREET_RE = re.compile(rf"\b(?:[A-Za-z]+|\d{{1,3}}(?:st|nd|rd|th)?)\s+{STREET_WORDS}\b", re.I)
SKIP_FIRST = {"the", "a", "an", "my", "our", "this", "that", "every", "same", "main", "next"}
BOROUGHS = ["manhattan", "brooklyn", "queens", "bronx", "staten island"]
SEVERE = ["waist", "submerged", "stuck", "trapped", "emergency", "rescue", "cars", "river"]
TYPE_KW = [
    ("flood", r"flood|water|submerged|inundat|river|sewer|basement|drain"),
    ("power_outage", r"power|electric|outage|blackout|grid|transformer|substation"),
    ("road_blockage", r"road|bridge|highway|tree|branch|blocked|debris|lane"),
]
SOP = {   # PLACEHOLDER text. Real version = RAG over NYC protocol documents.
    "flood": "SOP-FLOOD-01 (placeholder): dispatch DEP crew, close affected road segment",
    "power_outage": "SOP-POWER-01 (placeholder): notify utility, flag dependent traffic signals",
    "road_blockage": "SOP-BLOCK-01 (placeholder): notify DOT, reroute emergency vehicles",
    "unspecified": "SOP-GENERAL-01 (placeholder): route to operator for review",
}

def extract_entities(text):
    location = None
    m = ADDRESS_RE.search(text)
    if m:
        location = m.group(0)
    else:
        for m in STREET_RE.finditer(text):
            if m.group(0).split()[0].lower() not in SKIP_FIRST:
                location = m.group(0)
                break
    borough = next((b.title() for b in BOROUGHS if b in text.lower()), None)
    severity = "high" if any(w in text.lower() for w in SEVERE) else "medium"
    return location, borough, severity

def keyword_type(text):
    for label, pat in TYPE_KW:
        if re.search(pat, text, flags=re.I):
            return label
    return "unspecified"

In [4]:
import spacy
nlp = spacy.load("en_core_web_sm")
NOT_PLACES = ("hurricane", "harvey", "irma", "maria", "matthew", "idai", "storm", "youtube", "twitter")

def extract_entities(text):
    places = [e.text for e in nlp(text).ents
              if e.label_ in ("GPE", "LOC", "FAC")
              and not any(w in e.text.lower() for w in NOT_PLACES)]
    street = None
    m = ADDRESS_RE.search(text)
    if m:
        street = m.group(0)
    else:
        for m in STREET_RE.finditer(text):
            if m.group(0).split()[0].lower() not in SKIP_FIRST:
                street = m.group(0)
                break
    parts = ([street] if street else []) + places
    location = ", ".join(dict.fromkeys(parts)) if parts else None
    borough = next((b.title() for b in BOROUGHS if b in text.lower()), None)
    severity = "high" if any(w in text.lower() for w in SEVERE) else "medium"
    return location, borough, severity

In [5]:
import glob, os, re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def chunk_file(path, min_words=60, max_words=200):
    lines = [l.strip() for l in open(path, encoding="utf-8", errors="ignore").read().split("\n") if l.strip()]
    chunks, cur = [], []
    for line in lines:
        cur.append(line)
        if len(" ".join(cur).split()) >= min_words:
            chunks.append(" ".join(cur)); cur = []
    if cur and len(" ".join(cur).split()) >= 20:
        chunks.append(" ".join(cur))
    out = []
    for c in chunks:
        w = c.split()
        for i in range(0, len(w), max_words):
            out.append(" ".join(w[i:i + max_words]))
    return out

docs = []
for f in sorted(glob.glob("data/raw/protocols/*.txt")):
    for n, c in enumerate(chunk_file(f)):
        docs.append({"id": f"{os.path.basename(f)}#passage-{n}", "text": c})
kb = pd.DataFrame(docs)
print(len(kb), "passages from", len(glob.glob("data/raw/protocols/*.txt")), "files")
print(kb["id"].str.split("#").str[0].value_counts())

vec_kb = TfidfVectorizer(ngram_range=(1, 2), stop_words="english", sublinear_tf=True)
M = vec_kb.fit_transform(kb["text"])

COVERED = {"flood"}   # hazard types our saved documents actually cover
QUERY_BY_TYPE = {"flood": "flooding street sewer catch basin stormwater what residents should do during a storm"}

def attach_guidance(hazard_type, text="", min_score=0.12):
    if hazard_type not in COVERED:
        return None                                  # no documents for this hazard
    q = QUERY_BY_TYPE[hazard_type] + " " + text
    s = cosine_similarity(vec_kb.transform([q]), M)[0]
    i = int(s.argmax())
    if s[i] < min_score:
        return None
    return {"guidance_source": kb.iloc[i]["id"], "score": round(float(s[i]), 3)}

315 passages from 7 files
id
hmp_flooding.txt                    235
nycem_flooding.txt                   42
dep_plan_for_intense_storms.txt      12
dep_prepare_for_the_storm.txt        10
dep_monitor_during_the_storm.txt      7
dep_intro.txt                         5
dep_recover_after_the_storm.txt       4
Name: count, dtype: int64


In [6]:
from sentence_transformers import SentenceTransformer
emb = SentenceTransformer("all-MiniLM-L6-v2")
KB_E = emb.encode(list(kb["text"]), normalize_embeddings=True, batch_size=32)

QUERY_BY_TYPE_EMB = {"flood": "flooding in the street or basement, sewer backup, catch basin"}

def attach_guidance_emb(hazard_type, text="", min_score=0.50):
    if hazard_type not in COVERED:
        return None
    q = text if text else QUERY_BY_TYPE_EMB[hazard_type]
    s = KB_E @ emb.encode([q], normalize_embeddings=True)[0]
    i = int(s.argmax())
    if s[i] < min_score:
        return None
    return {"guidance_source": kb.iloc[i]["id"], "score": round(float(s[i]), 3)}

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
CONF_311 = {"Sewer Backup": 0.7}   # about 66% confirmed in a 29-row manual check (rough)

def rule_confidence(descriptor):
    for key, c in CONF_311.items():
        if key.lower() in str(descriptor).lower():
            return c
    return 0.9   # default for other rules (assumption, not measured)


def node_id(text):
    return "INC-" + hashlib.md5(text.encode()).hexdigest()[:8].upper()

def node_from_311(row):
    label = safety_311(row["complaint_type"], row["descriptor"])
    if label == "non_hazard":
        return None
    return {
        "incident_id": node_id(str(row["unique_key"])),
        "source": "NYC_311", "method": "category_rule",
        "timestamp": str(row["created_date"]),
        "hazard_type": label, "severity": "medium", "confidence": rule_confidence(row["descriptor"]),
        "location_text": row["incident_address"] if pd.notna(row["incident_address"]) else None,
        "borough": str(row["borough"]).title() if pd.notna(row["borough"]) else None,
        "lat": float(row["latitude"]) if pd.notna(row["latitude"]) else None,
        "lon": float(row["longitude"]) if pd.notna(row["longitude"]) else None,
        "guidance": attach_guidance_emb(label), "status": "unverified",
    }

def node_from_text(text, source="SOCIAL_DEMO", timestamp="2021-09-01T22:00:00"):
    p = safety_text(text)
    if p < THRESHOLD:
        return None
    loc, borough, severity = extract_entities(text)
    htype = keyword_type(text)
    return {
        "incident_id": node_id(text),
        "source": source, "method": "text_model",
        "timestamp": timestamp,
        "hazard_type": htype, "severity": severity, "confidence": round(p, 2),
        "location_text": loc, "borough": borough, "lat": None, "lon": None,
        "guidance": attach_guidance_emb(htype, text), "status": "unverified",
    }

In [8]:
n311 = pd.read_csv("data/processed/nyc311_ida_labeled.csv")

print("=== PATH 1: structured 311 reports ===")
for _, r in n311[n311["label"] != "non_hazard"].sample(2, random_state=5).iterrows():
    print(json.dumps(node_from_311(r), indent=2, default=str))
r = n311[n311["label"] == "non_hazard"].iloc[0]
print("Non-hazard 311 report ->", node_from_311(r), "(discarded)")

print("\n=== PATH 2: free text ===")
tests = [
    "water is rushing down 5th avenue in Manhattan, cars are stuck up to the doors",
    "huge tree fell across the road on Queens Blvd, nobody can pass",
    "my neighbor is playing loud music again",
    "basement flooded with sewage after the storm",
    "the street is a river near the subway entrance",
    "power is out on Atlantic Ave in Brooklyn, transformer sparking",
    "nice weather today, going for a walk",
    "big branch came down on Atlantic Ave and blocks the lane",
]
for t in tests:
    node = node_from_text(t)
    print("\n>", t)
    print(json.dumps(node, indent=2) if node else "  DISCARDED by Safety gate")

=== PATH 1: structured 311 reports ===
{
  "incident_id": "INC-8092BAA2",
  "source": "NYC_311",
  "method": "category_rule",
  "timestamp": "2021-09-01 11:25:00",
  "hazard_type": "flood",
  "severity": "medium",
  "confidence": 0.9,
  "location_text": null,
  "borough": "Staten Island",
  "lat": 40.561180032249545,
  "lon": -74.15583856604565,
  "guidance": {
    "guidance_source": "nycem_flooding.txt#passage-12",
    "score": 0.629
  },
  "status": "unverified"
}
{
  "incident_id": "INC-C708E6E1",
  "source": "NYC_311",
  "method": "category_rule",
  "timestamp": "2021-09-01 10:27:34",
  "hazard_type": "road_blockage",
  "severity": "medium",
  "confidence": 0.9,
  "location_text": "314 TRAVIS AVENUE",
  "borough": "Staten Island",
  "lat": 40.58767974707091,
  "lon": -74.15280046793022,
  "guidance": null,
  "status": "unverified"
}
Non-hazard 311 report -> None (discarded)

=== PATH 2: free text ===

> water is rushing down 5th avenue in Manhattan, cars are stuck up to the doors
 

In [9]:
nodes = []
for _, r in n311[n311["label"] != "non_hazard"].sample(5, random_state=5).iterrows():
    nodes.append(node_from_311(r))
for t in tests:
    n = node_from_text(t)
    if n:
        nodes.append(n)

with open("sample_incident_nodes.json", "w") as f:
    json.dump(nodes, f, indent=2, default=str)
print(len(nodes), "nodes saved to sample_incident_nodes.json")

9 nodes saved to sample_incident_nodes.json
